In [ ]:
import math
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.stats import spearmanr
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

# The `beta_ols` package lives next to this notebook in `train-notebooks/beta-training/`
BETA_TRAINING_DIR = CODE_DIR / "train-notebooks" / "beta-training"
if not (BETA_TRAINING_DIR / "beta_ols").is_dir():
    raise RuntimeError(f"`beta_ols` package not found in {BETA_TRAINING_DIR}")

# Run from `code/` and expose `code/`, `code/kev/`, and the beta-training folder on sys.path
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(BETA_TRAINING_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from data.base import build_long_records, fused_eval_context, group_records_by_state, make_view, predict_probs_grouped
from data.grummon2025 import (
    GRUMMON2025_PERSONA_COLS,
    TREAT_ARMS as GRUMMON2025_TREAT_ARMS,
    evaluate_grummon2025_ols,
)
from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_ols import core as beta_core
from beta_ols.unified_loss import build_sample_beta_operators, compute_unified_loss, make_respondent_item
from beta_ols.evaluation import build_population_eval, compare_population_betas, flat_probs_from_predictions

print(f"CODE_DIR: {CODE_DIR}")

# Datasets (loaded and replicated through `beta_ols`)

In [ ]:
# Load every dataset through its `beta_ols` module: task rows, regression rows, regression specs, human OLS replication
choice_dfs: dict[str, pd.DataFrame] = {}
regression_rows_by_dataset: dict[str, pd.DataFrame] = {}
regression_specs_by_dataset: dict[str, dict] = {}
human_ols_by_dataset: dict[str, pd.DataFrame] = {}

for dataset_name, dataset_module in beta_ols.DATASET_MODULES.items():
    choice_df = dataset_module.load_choice_df()
    rows = dataset_module.regression_rows(choice_df)
    specs = dataset_module.regression_specs(rows)
    choice_dfs[dataset_name] = choice_df
    regression_rows_by_dataset[dataset_name] = rows
    regression_specs_by_dataset[dataset_name] = specs
    human_ols_by_dataset[dataset_name] = beta_core.fit_human_ols(rows, specs)
    print(f"{dataset_name}: {choice_df['CASEID'].nunique():,} respondents | {len(choice_df):,} tasks | {len(specs)} regression(s)")
    display(beta_core.wide_ols_table(human_ols_by_dataset[dataset_name]).round(4))

# Every regression spec by group name (the unified loss looks specs up by group)
unified_specs = {}
for specs in regression_specs_by_dataset.values():
    unified_specs.update(specs)

maybe later explore the position effect and how we can replicate this too in the regressions

# Cross-dataset beta scale and sample-size target noise

In [ ]:
# Compare raw vs sigma_y-standardized human beta mean-square across datasets
scale_rows = []
for dataset_name, tidy in human_ols_by_dataset.items():
    choice_df = choice_dfs[dataset_name]
    option_counts = sorted({len(one_hot) for one_hot in choice_df["one_hot"]})
    sigma_by_model = tidy.groupby("model", sort=False)["sigma_y"].first()
    scale_rows.append(
        {
            "dataset": dataset_name,
            "respondents": choice_df["CASEID"].nunique(),
            "tasks": len(choice_df),
            "options_per_task": ", ".join(str(n) for n in option_counts),
            "regressions": len(sigma_by_model),
            "num_betas": len(tidy),
            "sigma_y": ", ".join(f"{sigma:.4g}" for sigma in sigma_by_model),
            "raw_signal_beta_ms": float((tidy["human_beta"] ** 2).mean()),
            "std_signal_beta_ms": float((tidy["std_beta"] ** 2).mean()),
        }
    )
display(pd.DataFrame(scale_rows).set_index("dataset").round(5))

In [ ]:
# Target noise of sample betas vs respondent sample size (Monte Carlo over respondents), per regression
SNR_DATASET = "grummon2023"
SNR_SAMPLE_SIZES = [16, 32, 64, 128, 256]
SNR_DRAWS = 4000

beta_noise_df = beta_core.beta_noise_by_sample_size(
    regression_rows_by_dataset[SNR_DATASET],
    regression_specs_by_dataset[SNR_DATASET],
    SNR_SAMPLE_SIZES,
    SNR_DRAWS,
    seed=42,
)
display(beta_noise_df.set_index(["model", "sample_size"]).round(4))

# Model, Grummon 2025 test evaluator, and unified respondent tokenization

In [ ]:
# Require CUDA and release any previously loaded `kev_model` before allocating a fresh 800M backbone
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required to run jaredpalmer/kev-0.8b in bf16 train mode")
if "kev_model" in globals():
    globals().pop("kev_model")
device = "cuda"
free_gpu()
torch.manual_seed(42)

# Build `jaredpalmer/kev-0.8b` with bf16 frozen backbone + rank-64 LoRA + pointer head
t_model0 = time.perf_counter()
kev_model = KevPointerModel.build(
    "jaredpalmer/kev-0.8b",
    device=device,
    lora_rank=64,
    weights_dtype="bf16",
)
kev_model.net.head.temperature = 1.0  # training uses raw uncalibrated temperature = 1.0
kev_model.train()                     # enable training mode (LoRA dropout active, trainable params require_grad)
torch.cuda.synchronize()
model_load_sec = time.perf_counter() - t_model0

trainable_params_count = sum(p.numel() for p in kev_model.trainable_parameters())
total_params_count = sum(p.numel() for p in kev_model.parameters())
display(
    pd.Series(
        {
            "base_model": "jaredpalmer/kev-0.8b",
            "mode": kev_model.mode,
            "training": kev_model.training,
            "weights_dtype": kev_model.meta.weights_dtype,
            "lora_rank": kev_model.meta.lora,
            "trainable_params": f"{trainable_params_count:,}",
            "total_params": f"{total_params_count:,}",
            "model_load_sec": round(model_load_sec, 3),
        },
        name="kev_800m_train_mode",
    ).to_frame()
)

In [ ]:
# Held-out Grummon 2025 evaluation: a state-grouped bundle for fast inference, plus population OLS operators built
# from `beta_ols.grummon2025` (the same item-level coding as its human replication and the training loss)
t_g25_bundle0 = time.perf_counter()
grummon2025_choice_df = choice_dfs["grummon2025"]
grummon2025_records = build_long_records(
    grummon2025_choice_df,
    persona_cols=GRUMMON2025_PERSONA_COLS,
    include_country=False,
    year="grummon2025",
)
grummon2025_eval_bundle = group_records_by_state(
    grummon2025_records,
    view_fn=make_view,
    name="grummon2025_primary_choices",
    valid=True,
)

# Human one-hot choices in bundle record order (used for the evaluation KL)
g25_y_3col_rows: list[list[float]] = []
for rec_idx, rec in enumerate(grummon2025_eval_bundle["records"]):
    h_dist = [float(v) for v in rec["human_dist"]]
    if len(h_dist) != 3 or sorted(h_dist) != [0.0, 0.0, 1.0]:
        raise ValueError(f"Grummon 2025 record {rec_idx}: expected binary 3-option one-hot human_dist, got {h_dist}")
    g25_y_3col_rows.append(h_dist)
y_grummon2025_3col = torch.tensor(g25_y_3col_rows, dtype=torch.float64)

# Population operators: every record is matched to its task by (CASEID, qid)
grummon2025_eval_ops = build_population_eval(
    beta_ols.DATASET_MODULES["grummon2025"],
    grummon2025_choice_df,
    regression_specs_by_dataset["grummon2025"],
    grummon2025_eval_bundle["records"],
)
grummon2025_human_betas = grummon2025_eval_ops["human_betas"]["grummon2025"]

# The data package's evaluator (person-mean form) must give the same human betas
_, _ref_g25_df = evaluate_grummon2025_ols(
    grummon2025_eval_bundle,
    [r["human_dist"] for r in grummon2025_eval_bundle["records"]],
)
_ref_g25_vec = torch.tensor([float(_ref_g25_df.loc[arm, "human"]) for arm in GRUMMON2025_TREAT_ARMS], dtype=torch.float64)
_max_g25_diff = float((grummon2025_human_betas - _ref_g25_vec).abs().max())
if _max_g25_diff > 1e-12:
    raise AssertionError(f"beta_ols Grummon 2025 OLS differs from evaluate_grummon2025_ols by {_max_g25_diff:.3e}")
g25_bundle_prep_sec = time.perf_counter() - t_g25_bundle0

print(
    f"Grummon 2025 test bundle: {len(grummon2025_eval_bundle['records']):,} tasks | "
    f"{len(grummon2025_eval_bundle['packed_views']):,} packed persona states | "
    f"max_diff vs evaluate_grummon2025_ols: {_max_g25_diff:.2e} | prep_sec: {g25_bundle_prep_sec:.3f}s"
)
display(
    pd.Series(
        grummon2025_human_betas.numpy(),
        index=regression_specs_by_dataset["grummon2025"]["grummon2025"].target_labels,
        name="grummon2025_human_pop_beta",
    ).round(4).to_frame()
)

In [ ]:
@torch.no_grad()
def validate_grummon2025_population_ols(
    model: KevPointerModel,
    eval_bundle: dict,
    eval_ops: dict,
    y_eval_3col: torch.Tensor,
) -> tuple[dict[str, float], pd.DataFrame]:
    # Time full evaluation wall-clock (LoRA fusion + CudaGraphs shared-state inference + population OLS)
    torch.cuda.synchronize()
    t_eval_wall0 = time.perf_counter()

    was_training = model.training
    model.eval()
    free_gpu()
    try:
        with torch.autocast("cuda", dtype=torch.bfloat16), fused_eval_context(model, eval_dtype=torch.bfloat16):
            preds, stats = predict_probs_grouped(model, eval_bundle, show_progress=True, return_stats=True)
    finally:
        # Always restore training mode and release temporary graph buffers before resuming training
        if was_training:
            model.train()
        free_gpu()

    if model.training != was_training:
        raise AssertionError(f"Expected model.training={was_training} after validation, got {model.training}")
    if any((not p.requires_grad) or p.numel() == 0 for p in model.trainable_parameters()):
        raise AssertionError("Trainable parameters were not cleanly restored after fused_eval_context")

    # 1. Evaluation KL over all 3-option tasks (bundle record order)
    t_ols0 = time.perf_counter()
    num_tasks = y_eval_3col.shape[0]
    if len(preds) != num_tasks:
        raise ValueError(f"Expected {num_tasks} Grummon 2025 task predictions, got {len(preds)}")
    pred_probs_3col = torch.from_numpy(np.stack(preds, axis=0)).to(dtype=torch.float64)
    if pred_probs_3col.shape != (num_tasks, 3):
        raise ValueError(f"Expected ({num_tasks}, 3) predicted probabilities, got {tuple(pred_probs_3col.shape)}")
    g25_kl_loss = float(
        torch.nn.functional.kl_div(
            pred_probs_3col.clamp_min(1e-12).log(),
            y_eval_3col,
            reduction="batchmean",
        )
    )

    # 2. Model vs human population betas with the module's own (item-level) coding
    probs_flat = flat_probs_from_predictions(eval_ops, preds)
    beta_metrics, beta_df = compare_population_betas(eval_ops, probs_flat)
    eval_ols_ms = (time.perf_counter() - t_ols0) * 1000.0
    eval_wall_sec = time.perf_counter() - t_eval_wall0

    metrics = {
        "g25_kl_loss": g25_kl_loss,
        "g25_beta_mse": beta_metrics["beta_mse"],
        "g25_beta_mae": beta_metrics["beta_mae"],
        "g25_beta_mape_pct": beta_metrics["beta_mape_pct"],
        "spearman_rho": beta_metrics["spearman_rho"],
        "eval_prep_sec": float(stats["prep_sec"]),
        "eval_gpu_sec": float(stats["gpu_sec"]),
        "eval_ols_ms": eval_ols_ms,
        "eval_wall_sec": eval_wall_sec,
        "eval_ms_per_task": (eval_wall_sec * 1000.0) / num_tasks,
    }
    return metrics, beta_df


# Run Step-0 untrained baseline evaluation on Grummon 2025 before any training
step0_val_metrics, step0_beta_df = validate_grummon2025_population_ols(
    kev_model,
    grummon2025_eval_bundle,
    grummon2025_eval_ops,
    y_grummon2025_3col,
)
display(pd.Series(step0_val_metrics, name="step_0_grummon2025_val").round(4).to_frame())
display(step0_beta_df.round(4))

In [ ]:
# Snapshot all untrained LoRA + pointer-head tensors on CPU so every training run starts identically
INIT_TRAINABLE_STATE = {
    name: param.detach().cpu().clone()
    for name, param in kev_model.named_parameters()
    if param.requires_grad
}


def reset_kev_trainable_weights(model: KevPointerModel, init_state: dict[str, torch.Tensor]) -> None:
    # Fail fast if trainable parameter names changed before restoring Step-0 weights onto GPU
    current_names = {name for name, param in model.named_parameters() if param.requires_grad}
    if current_names != set(init_state.keys()):
        raise ValueError("Trainable parameter names do not match INIT_TRAINABLE_STATE snapshot")
    with torch.no_grad():
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.copy_(init_state[name].to(device=param.device, dtype=param.dtype))
                param.grad = None
    model.train()
    free_gpu()


# Registry keyed by `beta_lambda` storing step/timing logs and Grummon 2025 test trajectories
experiment_runs: dict[float, dict] = {}
print(f"Snapshotted {len(INIT_TRAINABLE_STATE)} trainable tensors")

In [ ]:
# Pre-tokenize every respondent of every dataset once into training items:
# {dataset, caseid, enc, num_options, y_flat, outcome_rows}. Kev's prepared targets are checked against each beta part.
t_items0 = time.perf_counter()
unified_items: dict[str, list[dict]] = {}
for dataset_name, dataset_module in beta_ols.DATASET_MODULES.items():
    choice_df = choice_dfs[dataset_name]
    dataset_items = []
    respondents = beta_core.iter_respondents(choice_df)
    for caseid, r_df in tqdm(respondents, total=choice_df["CASEID"].nunique(), desc=dataset_name):
        dataset_items.append(make_respondent_item(kev_model.prepare, dataset_module, caseid, r_df))
    unified_items[dataset_name] = dataset_items
items_prep_sec = time.perf_counter() - t_items0

# Summary: respondents, tasks, and regression groups per dataset
items_summary_rows = []
for dataset_name, dataset_items in unified_items.items():
    task_count = 0
    for item in dataset_items:
        task_count += len(item["num_options"])
    items_summary_rows.append(
        {
            "dataset": dataset_name,
            "respondents": len(dataset_items),
            "tasks": task_count,
            "regression_groups": ", ".join(regression_specs_by_dataset[dataset_name]),
        }
    )
display(pd.DataFrame(items_summary_rows).set_index("dataset"))
print(f"item prep: {items_prep_sec:.1f}s")

In [ ]:
# Unified training: each optimization step draws one sample from ONE training dataset; every epoch covers every
# respondent of every dataset in `train_datasets` exactly once. Loss = `compute_unified_loss` (KL + beta_lambda * beta loss).
EPOCH_SEED = 42
BASE_LR_SAMPLE_1 = 2e-5               # validated 1-respondent/step base LR; scaled by sqrt(sample_size)


def scale_lr_for_sample_size(sample_size: int) -> float:
    # Square-root AdamW LR scaling keeps 1-epoch parameter travel comparable across respondent sample sizes
    if not isinstance(sample_size, int) or isinstance(sample_size, bool) or sample_size < 1:
        raise ValueError(f"sample_size must be a positive integer, got {sample_size!r}")
    return BASE_LR_SAMPLE_1 * math.sqrt(sample_size)


def split_into_chunks(items: list, chunk_size: int) -> list[list]:
    # Consecutive chunks of at most `chunk_size` respondents in original sample order
    return [items[start : start + chunk_size] for start in range(0, len(items), chunk_size)]


def predict_chunk(chunk_items: list[dict]) -> torch.Tensor:
    # Shared-prefix forward pass for one micro-chunk -> flat float64 logits: every option of every task, in item order
    with torch.autocast("cuda", dtype=torch.bfloat16):
        logits_per_respondent = kev_model.forward([it["enc"] for it in chunk_items], shared_prefix=True)
    flat_parts = []
    for item, respondent_logits in zip(chunk_items, logits_per_respondent, strict=True):
        option_counts = [int(question_logits.shape[0]) for question_logits in respondent_logits]
        if option_counts != item["num_options"]:
            raise ValueError(f"{item['caseid']}: logits option counts {option_counts} != expected {item['num_options']}")
        for question_logits in respondent_logits:
            flat_parts.append(question_logits.to(dtype=torch.float64))
    return torch.cat(flat_parts, dim=0)


def build_epoch_samples(
    rng: random.Random, train_datasets: list[str], sample_size_by_dataset: dict[str, int]
) -> list[tuple[str, list[dict]]]:
    # 1. Within each dataset: shuffle its respondents and cut them into samples of that dataset's sample size
    epoch_samples = []
    for dataset_name in train_datasets:
        dataset_items = list(unified_items[dataset_name])
        rng.shuffle(dataset_items)
        dataset_sample_size = sample_size_by_dataset[dataset_name]
        for sample_items in split_into_chunks(dataset_items, dataset_sample_size):
            epoch_samples.append((dataset_name, sample_items))
    # 2. Shuffle the order of all samples so datasets are interleaved across the epoch
    rng.shuffle(epoch_samples)
    return epoch_samples


def make_val_row(
    step: int,
    epoch: int,
    respondents_seen: int,
    train_loss: float,
    metrics: dict[str, float],
    lr: float,
    train_elapsed_sec: float,
) -> dict[str, float | int]:
    # Explicit schema: step/epoch counts, train loss, Grummon 2025 test metrics, LR, and eval/train timings
    return {
        "step": step,
        "epoch": epoch,
        "respondents_seen": respondents_seen,
        "train_loss": train_loss,
        "g25_kl_loss": metrics["g25_kl_loss"],
        "g25_beta_mse": metrics["g25_beta_mse"],
        "g25_beta_mae": metrics["g25_beta_mae"],
        "g25_beta_mape_pct": metrics["g25_beta_mape_pct"],
        "spearman_rho": metrics["spearman_rho"],
        "lr": lr,
        "eval_prep_sec": metrics["eval_prep_sec"],
        "eval_gpu_sec": metrics["eval_gpu_sec"],
        "eval_ols_ms": metrics["eval_ols_ms"],
        "eval_wall_sec": metrics["eval_wall_sec"],
        "eval_ms_per_task": metrics["eval_ms_per_task"],
        "train_elapsed_sec": train_elapsed_sec,
    }


def draw_live_plots(fig, axes, step_rows: list[dict], val_rows: list[dict], rolling_steps: int, steps_per_epoch: int) -> None:
    # Redraw train total loss per dataset, Grummon 2025 test KL / beta MSE / MAE / MAPE, and per-step LR
    for ax in axes.flat:
        ax.clear()
    val_steps = [r["step"] for r in val_rows]
    if step_rows:
        step_df_live = pd.DataFrame(step_rows)
        for dataset_name, dataset_steps in step_df_live.groupby("dataset"):
            rolling_loss = dataset_steps["total_loss"].rolling(rolling_steps, min_periods=1).mean()
            axes[0, 0].plot(dataset_steps["step"], rolling_loss, label=f"{dataset_name} (rolling {rolling_steps})")
        axes[0, 0].legend(fontsize=8)
        axes[1, 2].plot(step_df_live["step"], step_df_live["lr"])
    for ax, key, title in [
        (axes[0, 1], "g25_kl_loss", "Grummon 2025 KL loss"),
        (axes[0, 2], "g25_beta_mse", "Grummon 2025 beta MSE"),
        (axes[1, 0], "g25_beta_mae", "Grummon 2025 beta MAE"),
        (axes[1, 1], "g25_beta_mape_pct", "Grummon 2025 beta MAPE (%)"),
    ]:
        ax.plot(val_steps, [r[key] for r in val_rows], marker="o")
        ax.set_title(title)
    axes[0, 0].set_title("train total loss by dataset")
    axes[1, 2].set_title("learning rate")
    last_step = step_rows[-1]["step"] if step_rows else 0
    for ax in axes.flat:
        ax.set_xlabel("step")
        for b in range(steps_per_epoch, last_step + 1, steps_per_epoch):
            ax.axvline(b, color="gray", linestyle="--", linewidth=0.8)
    fig.tight_layout()


def run_unified_training(
    *,
    train_datasets: list[str],
    sample_size_by_dataset: dict[str, int],
    beta_lambda: float,
    lr: float,
    num_epochs: int,
    eval_every: int,
    lr_schedule: str,
    weight_decay: float,
    micro_chunk: int,
    gradient_checkpointing: bool,
) -> dict:
    # Fail fast on invalid training arguments
    if len(train_datasets) == 0 or len(set(train_datasets)) != len(train_datasets):
        raise ValueError(f"train_datasets must be a non-empty list without duplicates, got {train_datasets!r}")
    for dataset_name in train_datasets:
        if dataset_name not in unified_items:
            raise ValueError(f"Unknown training dataset {dataset_name!r}; expected one of {sorted(unified_items)}")
        dataset_sample_size = sample_size_by_dataset[dataset_name]
        if not isinstance(dataset_sample_size, int) or isinstance(dataset_sample_size, bool) or dataset_sample_size < 1:
            raise ValueError(f"sample size for {dataset_name} must be a positive integer, got {dataset_sample_size!r}")
    if not isinstance(num_epochs, int) or isinstance(num_epochs, bool) or num_epochs < 1:
        raise ValueError(f"num_epochs must be a positive integer, got {num_epochs!r}")
    if not isinstance(eval_every, int) or isinstance(eval_every, bool) or eval_every < 1:
        raise ValueError(f"eval_every must be a positive integer, got {eval_every!r}")
    if lr_schedule not in ("onecycle", "constant"):
        raise ValueError(f"lr_schedule must be 'onecycle' or 'constant', got {lr_schedule!r}")
    if weight_decay < 0.0:
        raise ValueError(f"weight_decay must be non-negative, got {weight_decay}")
    if not isinstance(micro_chunk, int) or isinstance(micro_chunk, bool) or micro_chunk < 1:
        raise ValueError(f"micro_chunk must be a positive integer, got {micro_chunk!r}")
    if not isinstance(gradient_checkpointing, bool):
        raise ValueError(f"gradient_checkpointing must be a bool, got {gradient_checkpointing!r}")
    if beta_lambda < 0.0:
        raise ValueError(f"beta_lambda must be non-negative, got {beta_lambda}")
    if lr <= 0.0:
        raise ValueError(f"lr must be positive, got {lr}")

    t_wall0 = time.perf_counter()

    # Reset weights to exact Step-0 initialization and seed RNGs so sample order matches across runs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    if gradient_checkpointing:
        kev_model.enable_checkpointing()
    else:
        kev_model.net.lm.gradient_checkpointing_disable()

    rng = random.Random(EPOCH_SEED)
    torch.manual_seed(EPOCH_SEED)
    torch.cuda.manual_seed_all(EPOCH_SEED)

    # Every epoch has the same number of samples: sum over datasets of ceil(respondents / sample size)
    steps_per_epoch = 0
    for dataset_name in train_datasets:
        steps_per_epoch += math.ceil(len(unified_items[dataset_name]) / sample_size_by_dataset[dataset_name])
    total_steps = steps_per_epoch * num_epochs
    postfix_every = max(1, steps_per_epoch // 30)
    rolling_steps = max(1, steps_per_epoch // 20)
    run_label = f"{'+'.join(train_datasets)} | lambda={beta_lambda:g}"

    try:
        trainable_params = list(kev_model.trainable_parameters())
        optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay, fused=True)
        if lr_schedule == "onecycle" and total_steps > 1:
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=lr,
                total_steps=total_steps,
                pct_start=0.1,
                div_factor=25.0,
                final_div_factor=1e4,
            )
        else:
            scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda _: 1.0)

        step_rows: list[dict] = []
        val_rows: list[dict] = [make_val_row(0, 0, 0, float("nan"), step0_val_metrics, float("nan"), 0.0)]
        val_display = display(pd.DataFrame(val_rows).set_index("step").round(5), display_id=True)
        fig, axes = plt.subplots(2, 3, figsize=(15, 7))
        plt.close(fig)
        draw_live_plots(fig, axes, step_rows, val_rows, rolling_steps, steps_per_epoch)
        plot_display = display(fig, display_id=True)
        final_beta_df = step0_beta_df
        cum_train_sec = 0.0
        cum_eval_sec = 0.0
        respondents_seen = 0

        pbar = tqdm(range(1, total_steps + 1), total=total_steps, desc=run_label)
        for step_idx in pbar:
            epoch_idx = (step_idx - 1) // steps_per_epoch + 1
            step_in_epoch = (step_idx - 1) % steps_per_epoch + 1
            if step_in_epoch == 1:
                epoch_samples = build_epoch_samples(rng, train_datasets, sample_size_by_dataset)
                if len(epoch_samples) != steps_per_epoch:
                    raise AssertionError(f"Expected {steps_per_epoch} samples per epoch, got {len(epoch_samples)}")
            sample_dataset, sample_items = epoch_samples[step_in_epoch - 1]
            n_sample_resp = len(sample_items)
            respondents_seen += n_sample_resp

            torch.cuda.synchronize()
            t_step0 = time.perf_counter()
            optimizer.zero_grad(set_to_none=True)
            sample_ops = build_sample_beta_operators(sample_items, unified_specs, device)
            sample_chunks = split_into_chunks(sample_items, micro_chunk)

            # 1. Pass 1: forward every micro-chunk without an autograd graph; record per-chunk CUDA RNG state
            chunk_rng_states = []
            chunk_logits_list = []
            with torch.no_grad():
                for chunk in sample_chunks:
                    chunk_rng_states.append(torch.cuda.get_rng_state())
                    chunk_logits_list.append(predict_chunk(chunk))
            flat_logits = torch.cat(chunk_logits_list, dim=0)
            torch.cuda.synchronize()
            t_after_fwd = time.perf_counter()

            # 2. Full-sample unified loss on the detached leaf `flat_logits`; backward fills d loss / d logit
            flat_logits.requires_grad_(True)
            loss_out = compute_unified_loss(flat_logits, sample_ops, beta_lambda)
            total_val = float(loss_out["total_loss"].detach())
            loss_out["total_loss"].backward()
            options_per_chunk = [int(chunk_logits.shape[0]) for chunk_logits in chunk_logits_list]
            grad_per_chunk = flat_logits.grad.split(options_per_chunk, dim=0)
            torch.cuda.synchronize()
            t_after_loss = time.perf_counter()

            # 3. Pass 2: replay each micro-chunk with its exact dropout RNG state and accumulate weight gradients
            for chunk, rng_state, chunk_grad in zip(sample_chunks, chunk_rng_states, grad_per_chunk, strict=True):
                torch.cuda.set_rng_state(rng_state)
                chunk_logits = predict_chunk(chunk)
                chunk_logits.backward(chunk_grad)
                del chunk_logits

            # 4. Clip gradients, apply AdamW update, and advance LR schedule
            grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            if not bool(torch.isfinite(grad_norm)):
                raise FloatingPointError(f"Non-finite gradient norm {float(grad_norm)} at step {step_idx}")
            grad_norm_val = float(grad_norm.detach().cpu())
            step_lr = float(optimizer.param_groups[0]["lr"])
            optimizer.step()
            if step_idx < total_steps:
                scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            del chunk_logits_list, flat_logits, grad_per_chunk, sample_ops
            loss_out["total_loss"] = None
            torch.cuda.synchronize()
            t_after_bwd = time.perf_counter()

            step_ms = (t_after_bwd - t_step0) * 1000.0
            cum_train_sec += t_after_bwd - t_step0
            has_beta = len(loss_out["group_beta_mse"]) > 0
            step_row = {
                "step": step_idx,
                "epoch": epoch_idx,
                "dataset": sample_dataset,
                "num_respondents": n_sample_resp,
                "respondents_seen": respondents_seen,
                "has_beta": has_beta,
                "skipped_groups": len(loss_out["skipped_groups"]),
                "kl_loss": loss_out["kl_loss"],
                "raw_beta_loss": loss_out["beta_loss"],
                "weighted_beta_loss": beta_lambda * loss_out["beta_loss"] if has_beta else 0.0,
                "total_loss": total_val,
                "grad_norm": grad_norm_val,
                "lr": step_lr,
                "fwd_ms": (t_after_fwd - t_step0) * 1000.0,
                "loss_ms": (t_after_loss - t_after_fwd) * 1000.0,
                "bwd_opt_ms": (t_after_bwd - t_after_loss) * 1000.0,
                "step_ms": step_ms,
                "ms_per_respondent": step_ms / n_sample_resp,
            }
            for group, group_mse in loss_out["group_beta_mse"].items():
                step_row[f"beta_mse | {group}"] = group_mse
            step_rows.append(step_row)

            if step_idx % postfix_every == 0 or step_idx == total_steps:
                recent = step_rows[-rolling_steps:]
                recent_betas = [r["raw_beta_loss"] for r in recent if r["has_beta"]]
                pbar.set_postfix(
                    kl=f"{float(np.mean([r['kl_loss'] for r in recent])):.3f}",
                    beta=f"{float(np.mean(recent_betas)):.4f}" if recent_betas else "n/a",
                    step_ms=f"{float(np.mean([r['step_ms'] for r in recent])):.1f}",
                )

            # 5. Evaluate held-out Grummon 2025 population OLS every `eval_every` steps and at the final step
            if step_idx % eval_every == 0 or step_idx == total_steps:
                v_metrics, final_beta_df = validate_grummon2025_population_ols(
                    kev_model,
                    grummon2025_eval_bundle,
                    grummon2025_eval_ops,
                    y_grummon2025_3col,
                )
                cum_eval_sec += v_metrics["eval_wall_sec"]
                previous_eval_step = val_rows[-1]["step"]
                steps_since_previous_eval = [r for r in step_rows if r["step"] > previous_eval_step]
                interval_train_loss = float(np.mean([r["total_loss"] for r in steps_since_previous_eval]))
                val_rows.append(
                    make_val_row(step_idx, epoch_idx, respondents_seen, interval_train_loss, v_metrics, step_lr, cum_train_sec)
                )
                val_display.update(pd.DataFrame(val_rows).set_index("step").round(5))

            if step_idx % postfix_every == 0 or step_idx % eval_every == 0 or step_idx == total_steps:
                draw_live_plots(fig, axes, step_rows, val_rows, rolling_steps, steps_per_epoch)
                plot_display.update(fig)
    finally:
        for param in kev_model.trainable_parameters():
            param.grad = None
        optimizer = scheduler = trainable_params = None
        chunk_logits_list = chunk_logits = flat_logits = grad_per_chunk = sample_ops = loss_out = grad_norm = None
        free_gpu()

    total_wall_sec = time.perf_counter() - t_wall0
    step_df = pd.DataFrame(step_rows)
    val_history_df = pd.DataFrame(val_rows).set_index("step")

    # Per-dataset train losses over the first and last 15% of that dataset's steps
    per_dataset_rows = []
    for dataset_name in train_datasets:
        dataset_steps = step_df[step_df["dataset"] == dataset_name]
        beta_steps = dataset_steps[dataset_steps["has_beta"]]
        edge_window = max(1, round(0.15 * len(dataset_steps)))
        if len(beta_steps) > 0:
            beta_edge_window = max(1, round(0.15 * len(beta_steps)))
            raw_beta_first15 = float(beta_steps["raw_beta_loss"].iloc[:beta_edge_window].mean())
            raw_beta_last15 = float(beta_steps["raw_beta_loss"].iloc[-beta_edge_window:].mean())
        else:
            raw_beta_first15 = float("nan")
            raw_beta_last15 = float("nan")
        per_dataset_rows.append(
            {
                "dataset": dataset_name,
                "sample_size": sample_size_by_dataset[dataset_name],
                "steps": len(dataset_steps),
                "steps_without_beta": int((~dataset_steps["has_beta"]).sum()),
                "skipped_groups_total": int(dataset_steps["skipped_groups"].sum()),
                "train_kl_first15pct": float(dataset_steps["kl_loss"].iloc[:edge_window].mean()),
                "train_kl_last15pct": float(dataset_steps["kl_loss"].iloc[-edge_window:].mean()),
                "train_raw_beta_first15pct": raw_beta_first15,
                "train_raw_beta_last15pct": raw_beta_last15,
            }
        )
    per_dataset_df = pd.DataFrame(per_dataset_rows).set_index("dataset")

    summary = pd.Series(
        {
            "train_datasets": "+".join(train_datasets),
            "beta_lambda": beta_lambda,
            "lr": lr,
            "num_epochs": num_epochs,
            "steps_per_epoch": steps_per_epoch,
            "total_steps": total_steps,
            "eval_every": eval_every,
            "lr_schedule": lr_schedule,
            "weight_decay": weight_decay,
            "micro_chunk": micro_chunk,
            "gradient_checkpointing": gradient_checkpointing,
            "mean_grad_norm": float(step_df["grad_norm"].mean()),
            "mean_step_ms": float(step_df["step_ms"].mean()),
            "mean_ms_per_respondent": (cum_train_sec * 1000.0) / respondents_seen,
            "mean_eval_wall_sec": float(val_history_df["eval_wall_sec"].iloc[1:].mean()),
            "train_only_sec": cum_train_sec,
            "eval_only_sec": cum_eval_sec,
            "total_wall_sec": total_wall_sec,
        },
        name=f"{run_label}_summary",
    )
    val_display.update(val_history_df.round(5))
    display(per_dataset_df.round(5))

    return {
        "train_datasets": list(train_datasets),
        "sample_size_by_dataset": {name: sample_size_by_dataset[name] for name in train_datasets},
        "beta_lambda": beta_lambda,
        "lr": lr,
        "num_epochs": num_epochs,
        "eval_every": eval_every,
        "lr_schedule": lr_schedule,
        "weight_decay": weight_decay,
        "micro_chunk": micro_chunk,
        "gradient_checkpointing": gradient_checkpointing,
        "summary": summary,
        "per_dataset_df": per_dataset_df,
        "step_df": step_df,
        "val_history_df": val_history_df,
        "final_beta_df": final_beta_df,
    }


print("Defined scale_lr_for_sample_size(sample_size) and run_unified_training(...)")

# Test micro-batching

In [ ]:
# Gradient check: verify two-pass micro-chunking (Method B) matches one full-sample backward (Method A) on the unified loss
CHECK_DATASET = "grummon2023"   # any key of `unified_items`
CHECK_SAMPLE_SIZE = 15          # respondents in the check sample (one graph of this size must fit for Method A)
CHUNK = 3                       # respondents per micro-chunk
LAM = 5.0                       # beta_lambda > 0 so both the KL and the beta gradients are tested

reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
kev_model.enable_checkpointing()
params = list(kev_model.trainable_parameters())

# Draw the check sample with a fixed seed and make sure every regression group is usable (full rank) in it
check_rng = random.Random(0)
sample = check_rng.sample(unified_items[CHECK_DATASET], CHECK_SAMPLE_SIZE)
check_ops = build_sample_beta_operators(sample, unified_specs, device)
if check_ops["skipped_groups"]:
    raise ValueError(f"Check sample is rank-deficient for {check_ops['skipped_groups']}; raise CHECK_SAMPLE_SIZE")
chunks = split_into_chunks(sample, CHUNK)


def collect_weight_gradients() -> torch.Tensor:
    # Flatten every trainable parameter gradient into one float32 vector, then clear `.grad`
    flat = torch.cat([p.grad.flatten().float() for p in params])
    for p in params:
        p.grad = None
    return flat


def method_b() -> tuple[float, torch.Tensor, list[torch.Tensor]]:
    # Two-pass micro-chunked forward/backward capturing per-chunk CUDA RNG states exactly as training does
    saved_states: list[torch.Tensor] = []
    logits_parts: list[torch.Tensor] = []
    with torch.no_grad():
        for chunk in chunks:
            saved_states.append(torch.cuda.get_rng_state())
            logits_parts.append(predict_chunk(chunk))
    all_logits = torch.cat(logits_parts, dim=0)

    all_logits.requires_grad_(True)
    loss = compute_unified_loss(all_logits, check_ops, LAM)["total_loss"]
    loss.backward()
    options_per_chunk = [int(part.shape[0]) for part in logits_parts]
    grad_per_chunk = all_logits.grad.split(options_per_chunk, dim=0)

    for chunk, rng_state, chunk_grad in zip(chunks, saved_states, grad_per_chunk, strict=True):
        torch.cuda.set_rng_state(rng_state)
        chunk_logits = predict_chunk(chunk)
        chunk_logits.backward(chunk_grad)
    return float(loss.detach()), collect_weight_gradients(), saved_states


def method_a(saved_states: list[torch.Tensor]) -> tuple[float, torch.Tensor]:
    # Single full-sample autograd graph replaying the exact per-chunk dropout RNG states from `method_b`
    logits_parts: list[torch.Tensor] = []
    for chunk, rng_state in zip(chunks, saved_states, strict=True):
        torch.cuda.set_rng_state(rng_state)
        logits_parts.append(predict_chunk(chunk))
    all_logits = torch.cat(logits_parts, dim=0)
    loss = compute_unified_loss(all_logits, check_ops, LAM)["total_loss"]
    loss.backward()
    return float(loss.detach()), collect_weight_gradients()


try:
    loss_b, grad_b, saved_rng_states = method_b()
    loss_a1, grad_a1 = method_a(saved_rng_states)
    loss_a2, grad_a2 = method_a(saved_rng_states)

    rel_a1_a2 = float((grad_a1 - grad_a2).norm() / grad_a1.norm())
    cos_a1_a2 = float(torch.nn.functional.cosine_similarity(grad_a1, grad_a2, dim=0))
    rel_a1_b = float((grad_a1 - grad_b).norm() / grad_a1.norm())
    cos_a1_b = float(torch.nn.functional.cosine_similarity(grad_a1, grad_b, dim=0))

    print(f"{CHECK_DATASET}: sample={len(sample)} respondents ({len(check_ops['num_options'])} tasks) | chunk={CHUNK} | lambda={LAM:g}")
    print(f"loss  A={loss_a1:.8f}  A again={loss_a2:.8f}  B={loss_b:.8f}")
    print(f"A vs A again (noise floor): rel={rel_a1_a2:.2e}  cos={cos_a1_a2:.8f}")
    print(f"A vs B       (two-pass):    rel={rel_a1_b:.2e}  cos={cos_a1_b:.8f}")
finally:
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)

# Find max micro chunk

In [ ]:
# Settings for the worst-case micro-chunk VRAM search over the longest respondents of ALL unified training datasets
SEARCH_GRADIENT_CHECKPOINTING = False
SEARCH_MAX_CHUNK = 64


def respondent_token_count(item: dict) -> int:
    # Total packed tokens in one respondent's encoding (persona prefix + every question branch)
    return len(item["enc"]["ids"])


all_unified_items = []
for dataset_items in unified_items.values():
    all_unified_items.extend(dataset_items)
respondents_by_size = sorted(all_unified_items, key=respondent_token_count, reverse=True)

largest_respondents_df = pd.DataFrame(
    {
        "dataset": [it["dataset"] for it in respondents_by_size[:10]],
        "caseid": [it["caseid"] for it in respondents_by_size[:10]],
        "num_tasks": [len(it["num_options"]) for it in respondents_by_size[:10]],
        "tokens": [respondent_token_count(it) for it in respondents_by_size[:10]],
    }
)
display(largest_respondents_df)

In [ ]:
def chunk_fits(chunk_size: int) -> tuple[bool, float]:
    # Test one Pass-2 forward + backward + fused AdamW step on the `chunk_size` longest Grummon 2023 respondents
    worst_chunk = respondents_by_size[:chunk_size]
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    torch.cuda.reset_peak_memory_stats()

    optimizer = torch.optim.AdamW(list(kev_model.trainable_parameters()), lr=1e-8, fused=True)
    chunk_logits = None
    fits = True
    try:
        chunk_logits = predict_chunk(worst_chunk)
        chunk_logits.sum().backward()
        optimizer.step()
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        fits = False
    finally:
        chunk_logits = None
        optimizer = None
        free_gpu()

    peak_gib = torch.cuda.max_memory_allocated() / 2**30
    reset_kev_trainable_weights(kev_model, INIT_TRAINABLE_STATE)
    return fits, peak_gib


if SEARCH_GRADIENT_CHECKPOINTING:
    kev_model.enable_checkpointing()
else:
    kev_model.net.lm.gradient_checkpointing_disable()

total_vram_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
search_rows: list[dict] = []
largest_fit = 0
smallest_fail = None


def record_trial(chunk_size: int, fits: bool, peak_gib: float) -> None:
    search_rows.append(
        {
            "chunk_size": chunk_size,
            "tokens": sum(respondent_token_count(it) for it in respondents_by_size[:chunk_size]),
            "fits": fits,
            "peak_gib": round(peak_gib, 2) if fits else float("nan"),
            "peak_pct_of_vram": round(100.0 * peak_gib / total_vram_gib, 1) if fits else float("nan"),
        }
    )


# Phase 1: double chunk_size (1, 2, 4, ...) until OOM or `SEARCH_MAX_CHUNK`
chunk_size = 1
while chunk_size <= SEARCH_MAX_CHUNK:
    fits, peak_gib = chunk_fits(chunk_size)
    record_trial(chunk_size, fits, peak_gib)
    if not fits:
        smallest_fail = chunk_size
        break
    largest_fit = chunk_size
    chunk_size *= 2

# Phase 2: binary search between `largest_fit` and `smallest_fail`
if smallest_fail is not None:
    low = largest_fit
    high = smallest_fail
    while high - low > 1:
        middle = (low + high) // 2
        fits, peak_gib = chunk_fits(middle)
        record_trial(middle, fits, peak_gib)
        if fits:
            low = middle
        else:
            high = middle
    largest_fit = low

if largest_fit < 1:
    raise RuntimeError("Even chunk_size=1 ran out of VRAM; enable SEARCH_GRADIENT_CHECKPOINTING=True")

MAX_FITTING_MICRO_CHUNK = largest_fit
display(pd.DataFrame(search_rows).sort_values("chunk_size").set_index("chunk_size"))
print(
    f"gradient_checkpointing={SEARCH_GRADIENT_CHECKPOINTING} | total VRAM {total_vram_gib:.1f} GiB | "
    f"MAX_FITTING_MICRO_CHUNK = {MAX_FITTING_MICRO_CHUNK}"
    + ("" if smallest_fail is not None else f" (no OOM up to SEARCH_MAX_CHUNK={SEARCH_MAX_CHUNK})")
)

# Overfit first

In [ ]:
# Overfit sanity check: multiple passes over the training datasets with pure KL/CE, constant LR, and zero weight decay
OVERFIT_TRAIN_DATASETS = ["grummon2023"]
OVERFIT_SAMPLE_SIZE_BY_DATASET = {"grummon2023": 128}
OVERFIT_NUM_EPOCHS = 10
OVERFIT_LR = scale_lr_for_sample_size(128)
OVERFIT_EVAL_EVERY = math.ceil(len(unified_items["grummon2023"]) / 128)  # 1 Grummon 2025 eval per epoch
OVERFIT_MICRO_CHUNK = MAX_FITTING_MICRO_CHUNK
OVERFIT_GRADIENT_CHECKPOINTING = SEARCH_GRADIENT_CHECKPOINTING

overfit_run = run_unified_training(
    train_datasets=OVERFIT_TRAIN_DATASETS,
    sample_size_by_dataset=OVERFIT_SAMPLE_SIZE_BY_DATASET,
    beta_lambda=0.0,
    lr=OVERFIT_LR,
    num_epochs=OVERFIT_NUM_EPOCHS,
    eval_every=OVERFIT_EVAL_EVERY,
    lr_schedule="constant",
    weight_decay=0.0,
    micro_chunk=OVERFIT_MICRO_CHUNK,
    gradient_checkpointing=OVERFIT_GRADIENT_CHECKPOINTING,
)

# Train

In [ ]:
# Training datasets (keys of `unified_items`); each step draws one sample from one of them
TRAIN_DATASETS = ["wu2025", "berke2023", "grummon2023"]
# Respondents per sample, per dataset
SAMPLE_SIZE_BY_DATASET = {"wu2025": 128, "berke2023": 128, "grummon2023": 128}
# Number of full passes over every respondent of every training dataset
NUM_EPOCHS = 5
# Steps between held-out Grummon 2025 evaluations (a final evaluation always runs at the last step)
EVAL_EVERY = 31
# Learning-rate schedule ("onecycle" or "constant"), AdamW weight decay, and peak LR
LR_SCHEDULE = "onecycle"
WEIGHT_DECAY = 0.01
TRAIN_LR = scale_lr_for_sample_size(128)
# Micro-chunk size and checkpointing from the worst-case VRAM search above
MICRO_CHUNK = MAX_FITTING_MICRO_CHUNK
GRADIENT_CHECKPOINTING = SEARCH_GRADIENT_CHECKPOINTING

steps_per_epoch = 0
config_rows = []
for dataset_name in TRAIN_DATASETS:
    dataset_steps = math.ceil(len(unified_items[dataset_name]) / SAMPLE_SIZE_BY_DATASET[dataset_name])
    steps_per_epoch += dataset_steps
    config_rows.append(
        {
            "dataset": dataset_name,
            "respondents": len(unified_items[dataset_name]),
            "sample_size": SAMPLE_SIZE_BY_DATASET[dataset_name],
            "steps_per_epoch": dataset_steps,
        }
    )
display(pd.DataFrame(config_rows).set_index("dataset"))
display(
    pd.Series(
        {
            "num_epochs": NUM_EPOCHS,
            "steps_per_epoch": steps_per_epoch,
            "total_steps": steps_per_epoch * NUM_EPOCHS,
            "eval_every": EVAL_EVERY,
            "lr_schedule": LR_SCHEDULE,
            "weight_decay": WEIGHT_DECAY,
            "max_lr": round(TRAIN_LR, 7),
            "micro_chunk": MICRO_CHUNK,
            "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        },
        name="unified_train_config",
    ).to_frame()
)

In [ ]:
# Run `NUM_EPOCHS` passes over TRAIN_DATASETS with pure KL/CE baseline (`beta_lambda = 0.0`)
beta_lambda = 0.0
experiment_runs[beta_lambda] = run_unified_training(
    train_datasets=TRAIN_DATASETS,
    sample_size_by_dataset=SAMPLE_SIZE_BY_DATASET,
    beta_lambda=beta_lambda,
    lr=TRAIN_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over TRAIN_DATASETS with mild standardized OLS beta weight (`beta_lambda = 1.0`)
beta_lambda = 1.0
experiment_runs[beta_lambda] = run_unified_training(
    train_datasets=TRAIN_DATASETS,
    sample_size_by_dataset=SAMPLE_SIZE_BY_DATASET,
    beta_lambda=beta_lambda,
    lr=TRAIN_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over TRAIN_DATASETS with standardized OLS beta weight (`beta_lambda = 2.0`)
beta_lambda = 2.0
experiment_runs[beta_lambda] = run_unified_training(
    train_datasets=TRAIN_DATASETS,
    sample_size_by_dataset=SAMPLE_SIZE_BY_DATASET,
    beta_lambda=beta_lambda,
    lr=TRAIN_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over TRAIN_DATASETS with standardized OLS beta weight (`beta_lambda = 3.0`)
beta_lambda = 3.0
experiment_runs[beta_lambda] = run_unified_training(
    train_datasets=TRAIN_DATASETS,
    sample_size_by_dataset=SAMPLE_SIZE_BY_DATASET,
    beta_lambda=beta_lambda,
    lr=TRAIN_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over TRAIN_DATASETS with standardized OLS beta weight (`beta_lambda = 4.0`)
beta_lambda = 4.0
experiment_runs[beta_lambda] = run_unified_training(
    train_datasets=TRAIN_DATASETS,
    sample_size_by_dataset=SAMPLE_SIZE_BY_DATASET,
    beta_lambda=beta_lambda,
    lr=TRAIN_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over TRAIN_DATASETS with moderate standardized OLS beta weight (`beta_lambda = 5.0`)
beta_lambda = 5.0
experiment_runs[beta_lambda] = run_unified_training(
    train_datasets=TRAIN_DATASETS,
    sample_size_by_dataset=SAMPLE_SIZE_BY_DATASET,
    beta_lambda=beta_lambda,
    lr=TRAIN_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Run `NUM_EPOCHS` passes over TRAIN_DATASETS with strong standardized OLS beta weight (`beta_lambda = 20.0`)
beta_lambda = 20.0
experiment_runs[beta_lambda] = run_unified_training(
    train_datasets=TRAIN_DATASETS,
    sample_size_by_dataset=SAMPLE_SIZE_BY_DATASET,
    beta_lambda=beta_lambda,
    lr=TRAIN_LR,
    num_epochs=NUM_EPOCHS,
    eval_every=EVAL_EVERY,
    lr_schedule=LR_SCHEDULE,
    weight_decay=WEIGHT_DECAY,
    micro_chunk=MICRO_CHUNK,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
)

In [ ]:
# Compare train metrics, held-out Grummon 2025 test OLS/KL metrics, and timings across completed runs
if not experiment_runs:
    raise ValueError("Run at least one `run_unified_training` cell first so `experiment_runs` is populated")
ran_lambdas = sorted(experiment_runs.keys())

compare_rows = [
    {
        "run": "step_0 (untrained)",
        "train_datasets": "",
        "beta_lambda": float("nan"),
        "num_epochs": 0,
        "steps_per_epoch": 0,
        "g25_kl_loss": step0_val_metrics["g25_kl_loss"],
        "g25_beta_mse": step0_val_metrics["g25_beta_mse"],
        "g25_beta_mae": step0_val_metrics["g25_beta_mae"],
        "g25_beta_mape_pct": step0_val_metrics["g25_beta_mape_pct"],
        "spearman_rho": step0_val_metrics["spearman_rho"],
        "mean_step_ms": float("nan"),
        "mean_ms_per_respondent": float("nan"),
        "mean_eval_wall_sec": step0_val_metrics["eval_wall_sec"],
        "train_only_sec": 0.0,
        "eval_only_sec": step0_val_metrics["eval_wall_sec"],
        "total_wall_sec": step0_val_metrics["eval_wall_sec"],
    }
]
for lam in ran_lambdas:
    run = experiment_runs[lam]
    s = run["summary"]
    final_val = run["val_history_df"].iloc[-1]
    compare_rows.append(
        {
            "run": f"lambda={lam:g}",
            "train_datasets": s["train_datasets"],
            "beta_lambda": lam,
            "num_epochs": int(s["num_epochs"]),
            "steps_per_epoch": int(s["steps_per_epoch"]),
            "g25_kl_loss": float(final_val["g25_kl_loss"]),
            "g25_beta_mse": float(final_val["g25_beta_mse"]),
            "g25_beta_mae": float(final_val["g25_beta_mae"]),
            "g25_beta_mape_pct": float(final_val["g25_beta_mape_pct"]),
            "spearman_rho": float(final_val["spearman_rho"]),
            "mean_step_ms": float(s["mean_step_ms"]),
            "mean_ms_per_respondent": float(s["mean_ms_per_respondent"]),
            "mean_eval_wall_sec": float(s["mean_eval_wall_sec"]),
            "train_only_sec": float(s["train_only_sec"]),
            "eval_only_sec": float(s["eval_only_sec"]),
            "total_wall_sec": float(s["total_wall_sec"]),
        }
    )

comparison_df = pd.DataFrame(compare_rows).set_index("run")
display(comparison_df.round(5))

# Per-dataset train KL and beta loss (first vs last 15% of that dataset's steps) across runs
per_dataset_frames = []
for lam in ran_lambdas:
    run_per_dataset = experiment_runs[lam]["per_dataset_df"].copy()
    run_per_dataset.insert(0, "beta_lambda", lam)
    per_dataset_frames.append(run_per_dataset.reset_index())
display(pd.concat(per_dataset_frames, ignore_index=True).set_index(["beta_lambda", "dataset"]).round(5))

# Compare the 5 predicted Grummon 2025 FOPL arm ATEs against ground-truth human RCT ATEs
betas_compare_cols = {
    "human_pop_beta": step0_beta_df["human_pop_beta"],
    "step_0": step0_beta_df["model_pop_beta"],
}
for lam in ran_lambdas:
    betas_compare_cols[f"lambda={lam:g}"] = experiment_runs[lam]["final_beta_df"]["model_pop_beta"]
grummon2025_betas_compare_df = pd.DataFrame(betas_compare_cols)
display(grummon2025_betas_compare_df.round(4))

In [ ]:
# Decompose the 5-term Grummon 2025 `g25_beta_mse` (`(1/5) * sum_k e_k^2`) arm-by-arm across completed runs
if not experiment_runs:
    raise ValueError("Run at least one `run_unified_training` cell first so `experiment_runs` is populated")

decomp_human = step0_beta_df["human_pop_beta"]
decomp_cols: dict[str, pd.Series] = {"human_beta": decomp_human}
sq_err_by_run: dict[str, pd.Series] = {}

for lam in sorted(experiment_runs.keys()):
    col_tag = f"lam_{lam:g}"
    pred_s = experiment_runs[lam]["final_beta_df"]["model_pop_beta"]
    if not pred_s.index.equals(decomp_human.index):
        raise ValueError(f"Beta index mismatch for run lambda={lam:g}")
    decomp_cols[f"pred_{col_tag}"] = pred_s
    sq_err_by_run[col_tag] = (pred_s - decomp_human) ** 2

for col_tag, sq_s in sq_err_by_run.items():
    decomp_cols[f"sq_err_{col_tag}"] = sq_s

grummon2025_mse_decomposition_df = pd.DataFrame(decomp_cols)
mean_row = pd.DataFrame(
    [{col: float(grummon2025_mse_decomposition_df[col].mean()) for col in grummon2025_mse_decomposition_df.columns}],
    index=pd.Index(["MEAN (= g25_beta_mse on sq_err cols)"], name="grummon2025_arm"),
)
display(pd.concat([grummon2025_mse_decomposition_df, mean_row]).round(5))